# Lab 2.4 — Route by Complexity

**Before you start:** select **Cell > Run All** to initialize the harness.

Cells marked `# ── YOUR WORK ──` are the ones you edit.

In [ ]:
# ── Harness setup (run once) ──────────────────────────────────────────────────
import sys, os, json, pathlib, time

sys.path.insert(0, '/opt/ara/lib')
from tina.client import llm_client, model_fast, model_strong
from elasticsearch import Elasticsearch

env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

client = llm_client()
FAST   = model_fast()
STRONG = model_strong()
es = Elasticsearch(os.environ['ES_URL'], api_key=os.environ['ES_API_KEY'], request_timeout=60)
EMBED_ID = os.environ.get('ARA_EMBED_ID', '.jina-embeddings-v5-text-small')
TRACES = pathlib.Path('/home/elastic/.traces')
TRACES.mkdir(parents=True, exist_ok=True)
print(f'Harness ready. FAST={FAST}, STRONG={STRONG}')

In [ ]:
# ── YOUR WORK ── Implement pick_tier ────────────────────────────────────────
STRONG_CUES = {'compare', 'analyze', 'explain', 'difference', 'differences', 'versus',
               'both', 'why', 'how does', 'interacts', 'interact', 'distinguish'}

def pick_tier(query: str) -> str:
    """Return 'fast' for single-hop lookups, 'strong' for multi-part analysis."""
    q = query.lower()
    words = q.split()
    # Length signal
    if len(words) > 20:
        return 'strong'
    # Conjunction signal
    conjunctions = q.count(' and ') + q.count(', ') + max(0, q.count('?') - 1)
    if conjunctions >= 2:
        return 'strong'
    # Cue word signal
    for cue in STRONG_CUES:
        if cue in q:
            return 'strong'
    return 'fast'

# Test on dev set
import json, pathlib
dev = json.loads(pathlib.Path('/home/elastic/dev-sets/dev-routing-queries.json').read_text())
correct = sum(1 for q in dev if pick_tier(q['text']) == q['label'])
print(f'Dev accuracy: {correct}/{len(dev)}')
for q in dev:
    tier = pick_tier(q['text'])
    ok = '✓' if tier == q['label'] else '✗'
    print(f'  {ok} [{tier}] {q["text"][:60]}')

In [ ]:
# Dispatch dev set through route A and save traces
import json, pathlib, time
from openai import OpenAI

FAST_MODEL  = os.environ.get('ARA_MODEL_FAST', 'gemini-2.5-flash')
STRONG_MODEL = os.environ.get('ARA_MODEL_STRONG', 'gemini-3.1-pro')
API_BASE = os.environ.get('ARA_OPENAI_API_BASE', os.environ.get('LLM_PROXY_URL', ''))
API_KEY  = os.environ.get('ARA_API_KEY', os.environ.get('LLM_APIKEY', ''))
proxy_client = OpenAI(base_url=API_BASE, api_key=API_KEY)

dev = json.loads(pathlib.Path('/home/elastic/dev-sets/dev-routing-queries.json').read_text())
trace_file = pathlib.Path('/home/elastic/.traces/routing-traces.jsonl')

for q in dev:
    tier  = pick_tier(q['text'])
    model = FAST_MODEL if tier == 'fast' else STRONG_MODEL
    try:
        resp = proxy_client.chat.completions.create(
            model=model, messages=[{'role': 'user', 'content': q['text']}],
            temperature=0, max_tokens=200)
        answer = resp.choices[0].message.content or ''
        tok = resp.usage.total_tokens if resp.usage else 0
    except Exception as e:
        answer, tok = f'ERROR: {e}', 0
    trace = {'query': q['text'], 'tier': tier, 'model': model, 'answer': answer, 'tokens': tok}
    with open(trace_file, 'a') as f:
        f.write(json.dumps(trace) + '\n')
    label_ok = '✓' if tier == q['label'] else '✗'
    print(f'  {label_ok} [{tier}] {q["text"][:50]}')

print('Routing traces written. Select Check in the sidebar.')